In [3]:
import requests

# Get voting records for a specific case
response = requests.get(
    "https://oda.ft.dk/api/Afstemning",
    params={"$filter": "sagid eq 102903", "$expand": "Stemme"}
)

# Check the response before trying to parse JSON
print(f"Status Code: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Response Text (first 500 chars): {response.text[:500]}")

# Only try to parse JSON if we got a successful response
if response.status_code == 200:
    try:
        data = response.json()
        print(f"\nSuccess! Got {len(data.get('value', []))} records")
    except requests.exceptions.JSONDecodeError as e:
        print(f"\nJSON Decode Error: {e}")
        print(f"Full response text: {response.text}")
else:
    print(f"\nError: HTTP {response.status_code}")
    print(f"Response: {response.text}")

Status Code: 400
Content-Type: None
Response Text (first 500 chars): 

Error: HTTP 400
Response: 


In [4]:
import requests

# Try a simpler query first to test the endpoint
print("Test 1: Basic query without filters")
response = requests.get("https://oda.ft.dk/api/Afstemning")
print(f"Status Code: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")

if response.status_code == 200:
    data = response.json()
    print(f"Success! Got {len(data.get('value', []))} records")
    if data.get('value'):
        print(f"Sample record keys: {list(data['value'][0].keys())}")
else:
    print(f"Error: {response.text[:500]}\n")

# Test 2: Try the filter with proper URL encoding
print("\nTest 2: With filter (alternative syntax)")
response = requests.get(
    "https://oda.ft.dk/api/Afstemning",
    params={
        "$filter": "sagid eq 102903",
        "$top": "10"  # Limit results
    }
)
print(f"Status Code: {response.status_code}")
print(f"URL: {response.url}")  # See the actual URL being called

if response.status_code == 200:
    data = response.json()
    print(f"Success! Got {len(data.get('value', []))} records")
else:
    print(f"Error: {response.text[:500]}")

Test 1: Basic query without filters
Status Code: 200
Content-Type: application/json; charset=utf-8
Success! Got 100 records
Sample record keys: ['id', 'nummer', 'konklusion', 'vedtaget', 'kommentar', 'mødeid', 'typeid', 'sagstrinid', 'opdateringsdato']

Test 2: With filter (alternative syntax)
Status Code: 400
URL: https://oda.ft.dk/api/Afstemning?%24filter=sagid+eq+102903&%24top=10
Error: 


In [11]:
import requests
import urllib.parse
import time
from typing import Dict, List, Optional, Union, Any
from datetime import datetime, timedelta
import json

class DanishParliamentAPI:
    """
    Production-ready client for Danish Parliament Open Data API (oda.ft.dk)

    Features:
    - Comprehensive error handling
    - Automatic retry with exponential backoff
    - Built-in pagination support
    - Rate limiting protection
    - Complete type hints
    """

    def __init__(self, timeout: int = 30, retry_attempts: int = 3):
        """
        Initialize the API client.

        Args:
            timeout: Request timeout in seconds
            retry_attempts: Number of retry attempts for failed requests
        """
        self.base_url = "https://oda.ft.dk/api/"
        self.timeout = timeout
        self.retry_attempts = retry_attempts
        self.last_request_time = 0
        self.min_request_interval = 0.1  # Minimum 100ms between requests

    def _rate_limit(self) -> None:
        """Enforce rate limiting between requests."""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.min_request_interval:
            time.sleep(self.min_request_interval - elapsed)
        self.last_request_time = time.time()

    def _make_request(self, url: str) -> Dict[str, Any]:
        """
        Make HTTP request with retry logic and error handling.

        Args:
            url: Complete URL to request

        Returns:
            Parsed JSON response

        Raises:
            APIError: For various API errors
            NetworkError: For network-related errors
        """
        self._rate_limit()

        for attempt in range(self.retry_attempts):
            try:
                response = requests.get(url, timeout=self.timeout)

                # Handle different HTTP status codes
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 400:
                    raise APIError(
                        f"Invalid query parameters. Check $expand and $filter syntax. "
                        f"URL: {url}"
                    )
                elif response.status_code == 404:
                    if 'api/' in url and url.count('/') == 4:  # Entity not found
                        raise EntityNotFoundError(f"Entity not found: {url}")
                    else:  # Invalid ID
                        raise RecordNotFoundError(f"Record not found: {url}")
                elif response.status_code == 501:
                    raise UnsupportedOperationError(
                        "Write operations are not supported by this API"
                    )
                else:
                    response.raise_for_status()

            except requests.exceptions.Timeout:
                if attempt < self.retry_attempts - 1:
                    wait_time = (2 ** attempt) * 1  # Exponential backoff
                    time.sleep(wait_time)
                    continue
                raise NetworkError(f"Request timed out after {self.timeout} seconds")

            except requests.exceptions.ConnectionError:
                if attempt < self.retry_attempts - 1:
                    wait_time = (2 ** attempt) * 1
                    time.sleep(wait_time)
                    continue
                raise NetworkError("Connection error - check your internet connection")

            except requests.exceptions.RequestException as e:
                raise NetworkError(f"Request failed: {str(e)}")

    def _build_url(self, entity: str, **params) -> str:
        """
        Build properly encoded URL with OData parameters.

        Args:
            entity: Entity name (e.g., 'Sag', 'Aktør')
            **params: OData parameters

        Returns:
            Complete URL with encoded parameters
        """
        # Start with base URL and entity
        url = f"{self.base_url}{entity}"

        if not params:
            return url

        # Build query parameters with proper encoding
        query_parts = []
        for key, value in params.items():
            if value is not None:
                # Ensure $ parameters are properly encoded
                if key.startswith('$'):
                    encoded_key = urllib.parse.quote(key, safe='$')
                else:
                    encoded_key = key

                encoded_value = urllib.parse.quote(str(value), safe="$()'/,%")
                query_parts.append(f"{encoded_key}={encoded_value}")

        return f"{url}?{'&'.join(query_parts)}"

    def get_cases(self, top: int = 100, skip: int = 0, filter_expr: Optional[str] = None, 
                  expand: Optional[str] = None, select: Optional[str] = None,
                  orderby: Optional[str] = None) -> Dict[str, Any]:
        """
        Get parliamentary cases (Sag) with optional filtering and expansion.

        Args:
            top: Number of records to return (max 100)
            skip: Number of records to skip for pagination
            filter_expr: OData filter expression
            expand: Related entities to include
            select: Specific fields to return
            orderby: Sort order

        Returns:
            API response with case data

        Example:
            # Get recent climate legislation
            cases = api.get_cases(
                filter_expr="substringof('klima', titel)",
                expand="Sagskategori",
                top=50
            )
        """
        params = {'$top': min(top, 100), '$skip': skip}  # Enforce 100 record limit

        if filter_expr:
            params['$filter'] = filter_expr
        if expand:
            params['$expand'] = expand
        if select:
            params['$select'] = select
        if orderby:
            params['$orderby'] = orderby

        url = self._build_url('Sag', **params)
        return self._make_request(url)

    def get_actors(self, top: int = 100, skip: int = 0, filter_expr: Optional[str] = None,
                   expand: Optional[str] = None) -> Dict[str, Any]:
        """
        Get parliamentary actors (Aktør) - politicians, committees, ministries.

        Args:
            top: Number of records to return (max 100)
            skip: Number of records to skip for pagination
            filter_expr: OData filter expression
            expand: Related entities to include

        Returns:
            API response with actor data

        Example:
            # Find all politicians with 'Jensen' in name
            actors = api.get_actors(
                filter_expr="substringof('Jensen', navn)"
            )
        """
        params = {'$top': min(top, 100), '$skip': skip}

        if filter_expr:
            params['$filter'] = filter_expr
        if expand:
            params['$expand'] = expand

        url = self._build_url('Aktør', **params)
        return self._make_request(url)

    def get_voting_records(self, politician_name: str, limit: int = 1000) -> List[Dict[str, Any]]:
        """
        Get all voting records for a specific politician.

        Args:
            politician_name: Full name of politician
            limit: Maximum number of votes to return

        Returns:
            List of voting records with expanded details

        Example:
            votes = api.get_voting_records("Frank Aaen")
        """
        all_votes = []
        skip = 0
        batch_size = 100

        while len(all_votes) < limit and skip < 10000:  # Safety limit
            params = {
                '$expand': 'Afstemning,Aktør',
                '$filter': f"Aktør/navn eq '{politician_name}'",
                '$top': batch_size,
                '$skip': skip
            }

            url = self._build_url('Stemme', **params)
            response = self._make_request(url)

            votes = response.get('value', [])
            if not votes:
                break

            all_votes.extend(votes)
            skip += batch_size

        return all_votes[:limit]

    def get_recent_changes(self, entity: str = 'Sag', hours_back: int = 24) -> Dict[str, Any]:
        """
        Get recent changes to parliamentary data.

        Args:
            entity: Entity to check ('Sag', 'Aktør', 'Afstemning', etc.)
            hours_back: How many hours back to check

        Returns:
            Recent changes in the specified entity

        Example:
            # Check for cases updated in last 4 hours
            recent = api.get_recent_changes('Sag', hours_back=4)
        """
        cutoff_time = datetime.now() - timedelta(hours=hours_back)
        iso_time = cutoff_time.strftime('%Y-%m-%dT%H:%M:%S')

        params = {
            '$filter': f"opdateringsdato gt datetime'{iso_time}'",
            '$orderby': 'opdateringsdato desc',
            '$top': 100
        }

        url = self._build_url(entity, **params)
        return self._make_request(url)

    def get_voting_session_details(self, voting_id: int, expand_votes: bool = True) -> Dict[str, Any]:
        """
        Get detailed information about a voting session.

        Args:
            voting_id: ID of the voting session (Afstemning)
            expand_votes: Whether to include individual vote details

        Returns:
            Voting session with optional vote details
        """
        expand_parts = ['Møde']
        if expand_votes:
            expand_parts.append('Stemme/Aktør')

        params = {
            '$filter': f'id eq {voting_id}',
            '$expand': ','.join(expand_parts)
        }

        url = self._build_url('Afstemning', **params)
        response = self._make_request(url)

        if response.get('value'):
            return response['value'][0]
        else:
            raise RecordNotFoundError(f"Voting session {voting_id} not found")

    def search_documents(self, search_term: str, include_files: bool = False) -> Dict[str, Any]:
        """
        Search parliamentary documents by title.

        Args:
            search_term: Term to search for in document titles
            include_files: Whether to include file download URLs

        Returns:
            Matching documents
        """
        params = {
            '$filter': f"substringof('{search_term}', titel)",
            '$top': 100
        }

        if include_files:
            params['$expand'] = 'Fil'

        url = self._build_url('Dokument', **params)
        return self._make_request(url)

    def get_entity_count(self, entity: str) -> int:
        """
        Get total count of records in an entity.

        Args:
            entity: Entity name

        Returns:
            Total number of records
        """
        params = {
            '$inlinecount': 'allpages',
            '$top': 1
        }

        url = self._build_url(entity, **params)
        response = self._make_request(url)

        count_str = response.get('odata.count', '0')
        return int(count_str)


# Custom Exception Classes
class APIError(Exception):
    """Base exception for API errors."""
    pass

class NetworkError(APIError):
    """Network-related errors."""
    pass

class EntityNotFoundError(APIError):
    """Entity does not exist."""
    pass

class RecordNotFoundError(APIError):
    """Specific record does not exist."""
    pass

class UnsupportedOperationError(APIError):
    """Operation not supported by API."""
    pass


# Usage Examples
if __name__ == "__main__":
    # Initialize client
    api = DanishParliamentAPI()

    try:
        # Get recent cases
        print("Getting recent cases...")
        cases = api.get_cases(top=5)
        print(f"Found {len(cases['value'])} cases")

        # Search for climate legislation
        print("\nSearching for climate legislation...")
        climate_cases = api.get_cases(
            filter_expr="substringof('klima', titel)",
            top=10
        )
        print(f"Found {len(climate_cases['value'])} climate-related cases")

        # Get total case count
        print("\nGetting total case count...")
        total_cases = api.get_entity_count('Sag')
        print(f"Total cases in database: {total_cases:,}")

        # Get recent changes
        print("\nChecking recent changes...")
        recent = api.get_recent_changes('Sag', hours_back=24)
        print(f"Cases updated in last 24 hours: {len(recent['value'])}")

    except APIError as e:
        print(f"API Error: {e}")
    except NetworkError as e:
        print(f"Network Error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

Getting recent cases...
Found 5 cases

Searching for climate legislation...
Found 10 climate-related cases

Getting total case count...
Total cases in database: 97,045

Checking recent changes...
Cases updated in last 24 hours: 62


In [9]:
# Print all values of the first climate case
first_case = climate_cases['value'][0]
for key, value in first_case.items():
    print(f"{key}: {value}")

id: 116
typeid: 9
kategoriid: None
statusid: 17
titel: Folketinget bakker op om regeringens mål om, at Danmarks udledninger af drivhusgasser i 2020 reduceres med 40 pct. i forhold til niveauet i 1990. Folketinget noterer sig derfor med tilfredshed, at regeringen planlægger at fremsætte forslag til en klimalov i 2014. Klimaloven skal sikre monitorering, evaluering af klimaindsatsen og større offentlighed om klimabeslutninger.
 
Folketinget konstaterer, at finansloven for 2014 indeholder tiltag, der reducerer udledningen af drivhusgasser med mindst 50.000 ton CO2-ækvivalenter i 2020, og opfordrer regeringen til løbende at fremsætte forslag, der yderligere kan reducere udledningen af drivhusgasser med henblik på at nå målet om en 40 procents reduktion i 2020. Folketinget lægger vægt på, at klimaindsatsen skal skabe jobs og ikke koste jobs. Det skal ske ved at gøre brug af moderne teknologier, sikre synergieffekter i alle sektorer og ved at sikre, at det danske erhvervsliv forbliver konkur

In [ ]:
def get_voting_session_details(self, voting_id: int, expand_votes: bool = True) -> Dict[str, Any]:
        """
        Get detailed information about a voting session.

        Args:
            voting_id: ID of the voting session (Afstemning)
            expand_votes: Whether to include individual vote details

        Returns:
            Voting session with optional vote details
        """

In [15]:
api.get_voting_session_details(12)

{'Møde': {'id': 962,
  'titel': 'Møde i salen',
  'lokale': '',
  'nummer': '99',
  'dagsordenurl': '',
  'starttidsbemærkning': '',
  'offentlighedskode': 'O',
  'dato': '2014-06-11T09:00:00',
  'statusid': 2,
  'typeid': 1,
  'periodeid': 32,
  'opdateringsdato': '2014-09-19T14:36:51.117'},
 'Stemme': [{'Aktør': {'id': 5,
    'typeid': 5,
    'gruppenavnkort': None,
    'navn': 'Frank Aaen',
    'fornavn': 'Frank',
    'efternavn': 'Aaen',
    'biografi': '<member><url>/medlemmer/fhvmf/f/frank-aaen</url><status>0</status><formattedDateLongMonth>30. september 2022</formattedDateLongMonth><profession>Fhv. næstformand for Statsrevisorerne, fhv. MF, økonom</profession><firstname>Frank</firstname><lastname>Aaen</lastname><party>Enhedslisten</party><partyShortname>EL</partyShortname><title>Frank Aaen (EL)</title><sex>Mand</sex><born>25-07-1951</born><died/><educationStatistic>LVU</educationStatistic><occupationStatistic>Privat</occupationStatistic><pictureMiRes>https://www.ft.dk/-/media/cv

In [13]:
api.get_actors(filter_expr="substringof('Frederiksen', navn)")

{'odata.metadata': 'https://oda.ft.dk/api/$metadata#Akt%C3%B8r',
 'value': [{'id': 138,
   'typeid': 5,
   'gruppenavnkort': None,
   'navn': 'Mette Frederiksen',
   'fornavn': 'Mette',
   'efternavn': 'Frederiksen',
   'biografi': '<member><url/><status>1</status><sex>Kvinde</sex><educationStatistic>LVU</educationStatistic><occupationStatistic>Privat</occupationStatistic><title>Mette Frederiksen (S)</title><firstname>Mette</firstname><lastname>Frederiksen</lastname><profession>Statsminister</profession><party>Socialdemokratiet</party><partyShortname>S</partyShortname><formattedDateLongMonth/><born>19-11-1977</born><died/><pictureMiRes>https://www.ft.dk/-/media/cv/foto/2022/s/mette-frederiksen/mette_frederiksen_500.ashx</pictureMiRes><pictureHiRes>https://www.ft.dk/-/media/cv/foto/2022/s/mette-frederiksen/mette_frederiksen-fotograf_socialdemokratiet.zip</pictureHiRes><addresses><address>Statsministeriet, Christiansborg, Prins Jørgens Gård 11 1218&amp;nbsp;København K</address></address